#ML MODEL TO PREDICT PRICES


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import time
##from joblib import dump
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


In [2]:
df = pd.read_csv("ResaleData.csv")
print(df.columns)

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'remaining_lease', 'resale_price'],
      dtype='object')


In [3]:
#check if got NA values
#df.isna().sum()
df.isnull().sum()

month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

In [4]:
# drop streetname, lease, block -- not using these features
df.drop(['street_name', 'remaining_lease', 'block','storey_range', 'floor_area_sqm','flat_model','lease_commence_date'], axis=1, inplace=True)

In [5]:
df.head()

,month,town,flat_type,resale_price
0,2017-01,ANG MO KIO,2 ROOM,232000.0
1,2017-01,ANG MO KIO,3 ROOM,250000.0
2,2017-01,ANG MO KIO,3 ROOM,262000.0
3,2017-01,ANG MO KIO,3 ROOM,265000.0
4,2017-01,ANG MO KIO,3 ROOM,265000.0


In [10]:
# 1) Convert sale_date to datetime
df['month'] = pd.to_datetime(df['month'])

# 2) Extract year and month
df['year'] = df['month'].dt.year
df['month_num'] = df['month'].dt.month

# 3) Define features and target
categorical_features = ['town', 'flat_type']
numeric_features = ['year', 'month_num']
target = 'resale_price'

# 4) One-hot encode the categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(df[categorical_features])

# 5) Combine encoded categorical and numeric features into one array X
X = np.hstack((encoded_cats, df[numeric_features].values))

# 6) Define the target array y
y = df[target].values

# 7) Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8) Train a Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 9) Evaluate the model
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Evaluation:")
print("Mean Absolute Error:", mae)
print("R^2 Score:", r2)


Linear Regression Evaluation:
Mean Absolute Error: 76484.12780337011
R^2 Score: 0.6853270773936866


In [11]:
# Initialize and train the multivariate (multiple) linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Multivariate Linear Regression Evaluation:")
print("Mean Absolute Error:", mae)
print("R^2 Score:", r2)


Multivariate Linear Regression Evaluation:
Mean Absolute Error: 76484.12780337011
R^2 Score: 0.6853270773936866


In [26]:
def predict_future_price(model, encoder, categorical_features, numeric_features):
    # Prompt user for inputs
    town_input = input("Enter the town (e.g., 'QUEENSTOWN'): ")
    flat_type_input = input("Enter the flat type (e.g., '4 ROOM'): ")
    
    # Ensure valid integer input for the year
    while True:
        try:
            year_input = int(input("Enter the future year (e.g., 2025): "))
            break
        except ValueError:
            print("Please enter a valid integer for the year.")
    
    # Ensure valid integer input for the month
    while True:
        try:
            month_input = int(input("Enter the month as a number (1-12): "))
            if 1 <= month_input <= 12:
                break
            else:
                print("Month must be between 1 and 12.")
        except ValueError:
            print("Please enter a valid integer for the month.")
    
    # Create a DataFrame for the new data
    future_data = pd.DataFrame({
        'town': [town_input],
        'flat_type': [flat_type_input],
        'year': [year_input],
        'month_num': [month_input]
    })
    
    # One-hot encode the categorical part using the same encoder
    future_cat = future_data[categorical_features]
    encoded_future_cat = encoder.transform(future_cat)
    
    # Extract numeric features and combine with the encoded categorical features
    future_numeric = future_data[numeric_features].values
    X_future = np.hstack((encoded_future_cat, future_numeric))
    
    # Predict the resale price
    future_price_pred = model.predict(X_future)
    predicted_price = future_price_pred[0]
    
    print(f"\nPredicted resale price for a {flat_type_input} in {town_input} for {year_input}-{month_input}:")
    print(f"${predicted_price:,.2f}")
    
  


In [28]:
predict_future_price(
    model=model,
    encoder=encoder,
    categorical_features=categorical_features,
    numeric_features=numeric_features
)



Predicted resale price for a 3 ROOM  in ANG MO KIO for 2017-2:
$401,047.15
